# SNF Modeling, Evaluation, And Error Analysis

**Executive takeaway:** SNF payroll approval ranking should be evaluated by temporal approval-budget performance and estimated exposure captured, then compared directly with manual threshold baselines.

In [1]:
from common.plots import LetsPlot, aes, geom_bar, ggplot, labs, theme_minimal

from payroll_anomaly_ranking.columns import PayrollCol, ScoreCol
from payroll_anomaly_ranking.config import PayrollConfig
from payroll_anomaly_ranking.evaluation import evaluate_scores
from payroll_anomaly_ranking.pipeline import run_pipeline

LetsPlot.setup_html()

results = run_pipeline(
    PayrollConfig(employee_count=180, pay_periods=14, review_budgets=(10, 25, 50)),
)

## Temporal Approval-Budget Metrics

Training, baselines, and scoring use earlier pay periods to score later pay periods. Metrics focus on how much value administrators can capture inside realistic weekly review capacity.

In [2]:
results.metrics

k,precision_at_k,recall_at_k,f1_at_k,dollars_captured_at_k,dollar_capture_rate,average_anomaly_rank,mean_reciprocal_rank,pr_auc
f64,f64,f64,f64,f64,f64,f64,f64,f64
10.0,0.75,0.76087,0.755396,33608.43,0.785873,2.231884,0.649258,0.835634
25.0,0.385714,0.978261,0.553279,42013.47,0.98241,2.231884,0.649258,0.835634
50.0,0.197143,1.0,0.329356,42765.7,1.0,2.231884,0.649258,0.835634


## Model Comparison

The hybrid score combines deterministic rules, robust statistics, employee history, peer/facility normalization, schedule/timeclock context, premium eligibility, ML, and estimated exposure.

In [3]:
results.model_comparison

model,k,precision_at_k,recall_at_k,f1_at_k,average_anomaly_rank,mean_reciprocal_rank,pr_auc
str,f64,f64,f64,f64,f64,f64,f64
"""rule_score""",10.0,0.714286,0.724638,0.719424,15.594203,0.258879,0.791629
"""statistical_score""",10.0,0.435714,0.442029,0.438849,13.224638,0.141191,0.393946
"""schedule_timeclock_score""",10.0,0.25,0.253623,0.251799,22.282609,0.094564,0.230384
"""premium_eligibility_score""",10.0,0.0,0.0,0.0,578.007246,0.001815,0.007755
"""ml_score""",10.0,0.721429,0.731884,0.726619,7.384058,0.258717,0.829022
"""hybrid_score""",10.0,0.75,0.76087,0.755396,7.913043,0.27312,0.835634


## Manual Threshold Baseline Comparison

Threshold rules are easy to configure but can overflag legitimate staffing pressure while missing unsupported premium or paid-vs-scheduled exceptions. This table measures threshold review volume and exposure captured.

In [4]:
evaluate_scores(results.scored).threshold_baseline_metrics

baseline,review_volume,precision_at_k,recall_at_k,exposure_captured_at_k,dollars_captured_at_k,false_positives_avoided
str,i64,f64,f64,f64,f64,i64
"""gross_pay_threshold""",0,0.0,0.0,0.0,0.0,0
"""total_hours_threshold""",142,0.732394,0.753623,53230.202325,32664.22,38
"""overtime_hours_threshold""",142,0.732394,0.753623,53230.202325,32664.22,38
"""premium_dollars_threshold""",0,0.0,0.0,0.0,0.0,0
"""paid_vs_scheduled_threshold""",410,0.0,0.0,59386.863975,0.0,410


In [5]:
results.scored.select(
    [
        PayrollCol.FACILITY_ID,
        PayrollCol.ROLE,
        PayrollCol.SHIFT_TYPE,
        PayrollCol.GROSS_PAY,
        PayrollCol.PAID_HOURS,
        PayrollCol.OVERTIME_HOURS,
        PayrollCol.PREMIUM_PAY,
        ScoreCol.ESTIMATED_EXPOSURE,
        ScoreCol.FINAL_APPROVAL_EXCEPTION_SCORE,
    ],
).sort(ScoreCol.FINAL_APPROVAL_EXCEPTION_SCORE, descending=True).head(15)

facility_id,role,shift_type,gross_pay,paid_hours,overtime_hours,premium_pay,estimated_exposure,final_approval_exception_score
str,str,str,f64,f64,f64,f64,f64,f64
"""SNF-F006""","""RN""","""Double""",1067.49,17.65,9.65,44.88,858.71,1.0
"""SNF-F006""","""LPN""","""Double""",933.76,17.99,9.99,34.13,725.95,0.993298
"""SNF-F002""","""CNA""","""Double""",464.68,18.0,10.0,35.96,302.145,0.986239
"""SNF-F002""","""RN""","""Double""",842.48,17.57,9.57,18.92,634.67,0.979059
"""SNF-F002""","""CNA""","""Double""",455.67,17.94,9.94,28.63,273.402,0.971823
…,…,…,…,…,…,…,…,…
"""SNF-F006""","""RN""","""Double""",1169.61,17.5,9.5,28.63,961.86,0.92503
"""SNF-F001""","""Therapy""","""Double""",794.25,17.19,9.19,34.17,586.44,0.922836
"""SNF-F005""","""CNA""","""Double""",493.81,18.0,10.0,0.0,296.286,0.912292


In [6]:
category_errors = results.category_error_analysis
(
    ggplot(category_errors, aes(PayrollCol.ANOMALY_CATEGORY, "true_anomalies"))
    + geom_bar(stat="identity", fill="#7a4e9d")
    + labs(
        title="Synthetic SNF exception categories available for evaluation only",
        x="Evaluation-only category",
        y="Synthetic anomalies",
    )
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7f61c4895ae0>)

## What This Proves

The project evaluates the approval assistant as a ranked queue under constrained weekly review capacity, not as a generic classifier detached from administrator workflow.